# 02. Visual Embeddings

This notebook generates visual embeddings for the final strict dataset using pre-trained vision encoders. The goal is to transform each artwork image into a high-dimensional vector representation that can later be used in the modeling stage.

The first encoder used is SigLIP, a vision-language model trained to align images and text in a shared semantic space.

In [1]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

/home/agrupa-lab/agrupa/IE_capstones/Naji/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
DATASET_CSV = "../Data/Processed/final_dataset_strict.csv"
MODEL_NAME = "google/siglip-base-patch16-224"

OUTPUT_DIR = Path("../Embeddings/siglip")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_EMBEDDINGS = OUTPUT_DIR / "siglip_strict_embeddings.npy"
OUTPUT_IDS = OUTPUT_DIR / "siglip_strict_catnos.csv"
OUTPUT_MERGED = OUTPUT_DIR / "final_dataset_strict_with_siglip.csv"
OUTPUT_FAILED = OUTPUT_DIR / "siglip_strict_failed_paths.csv"

## 1. Loading the final strict dataset

The strict dataset created in the previous notebook is used as the main input for embedding extraction. Before generating embeddings, the notebook checks which image column is available and resolves the full image paths needed to load the files from disk.

In [15]:
df = pd.read_csv(DATASET_CSV)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head(2)

Dataset shape: (3198, 19)

Columns:
['cat_no', 'titulo', 'autor', 'escuela_obra', 'tipo_objeto', 'datacion', 'tema', 'is_religious', 'is_fauna', 'century', 'image_path', 'descripcion', 'animal_cluster', 'n_descriptores_fila', 'n_en_diccionario_fila', 'dirmean_Warmth', 'n_dirmean_Warmth', 'dirmean_Competence', 'n_dirmean_Competence']


,cat_no,titulo,autor,escuela_obra,tipo_objeto,datacion,tema,is_religious,is_fauna,century,image_path,descripcion,animal_cluster,n_descriptores_fila,n_en_diccionario_fila,dirmean_Warmth,n_dirmean_Warmth,dirmean_Competence,n_dirmean_Competence
0,P000002,El juicio de Paris,"Albani, Francesco",Italiana,Pintura,1650 - 1660,NaN,0,1,17th c.,obras/P000002.jpg,"La obra de Francesco Albani, uno de los discíp...",purity,890.0,62.0,0.333333,9,0.7,10
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",Italiana,Pintura,1584,NaN,1,1,16th c.,obras/P000006.jpg,"San José, la Virgen con el Niño en brazos y Sa...",other,148.0,17.0,0.937500,4,0.5,2


In [16]:
# ----------------------------
# Resolve image paths safely
# ----------------------------
# The dataset may contain either:
# - file_path: already absolute
# - image_path: relative path such as "obras/P000002.jpg"
# We handle both cases safely.

RAW_IMAGE_ROOT = Path("/home/agrupa-lab/agrupa/data_raw")

if "file_path" in df.columns:
    df["resolved_image_path"] = df["file_path"].astype(str)
elif "image_path" in df.columns:
    df["resolved_image_path"] = df["image_path"].apply(
        lambda p: str(RAW_IMAGE_ROOT / str(p)) if pd.notna(p) else None
    )
else:
    raise ValueError("No image column found. Expected 'file_path' or 'image_path'.")

df["file_exists"] = df["resolved_image_path"].apply(lambda p: Path(p).exists() if pd.notna(p) else False)

print("Total rows:", len(df))
print("Missing images:", int((~df["file_exists"]).sum()))

df = df[df["file_exists"]].copy().reset_index(drop=True)
print("Rows used for embeddings:", len(df))

Total rows: 3198
Missing images: 138
Rows used for embeddings: 3060


## 2. Loading the SigLIP model

SigLIP is loaded in evaluation mode and used only as a frozen feature extractor. No fine-tuning is applied at this stage; the objective is to obtain stable pre-trained image representations for later probing and regression experiments.

In [17]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 13807.18it/s]


SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features

In [18]:
def load_image(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

## 3. Extracting embeddings in batches

Embeddings are generated in batches to improve efficiency and reduce memory issues. For each image, the output vector is stored together with its corresponding `cat_no`, so that the embeddings can later be aligned back to the dataset used for modeling.

In [19]:
all_embeddings = []
all_catnos = []
failed_paths = []

for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Extracting SigLIP"):
    batch = df.iloc[i:i + BATCH_SIZE]

    images = []
    catnos = []

    for _, row in batch.iterrows():
        img = load_image(row["resolved_image_path"])
        if img is None:
            failed_paths.append(row["resolved_image_path"])
            continue
        images.append(img)
        catnos.append(row["cat_no"])

    if len(images) == 0:
        continue

    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.get_image_features(**inputs)

    if hasattr(out, "image_embeds"):
        emb = out.image_embeds
    elif hasattr(out, "pooler_output"):
        emb = out.pooler_output
    elif hasattr(out, "last_hidden_state"):
        emb = out.last_hidden_state[:, 0, :]
    else:
        emb = out

    emb = emb.detach().cpu().numpy()

    all_embeddings.append(emb)
    all_catnos.extend(catnos)

if len(all_embeddings) == 0:
    raise RuntimeError("No embeddings extracted.")

embeddings = np.vstack(all_embeddings)
ids_df = pd.DataFrame({"cat_no": all_catnos})

print("Embedding shape:", embeddings.shape)
print("IDs count:", len(ids_df))
print("Failed images:", len(failed_paths))

Extracting SigLIP: 100%|██████████| 192/192 [00:28<00:00,  6.69it/s]

Embedding shape: (3060, 768)
IDs count: 3060
Failed images: 0


In [20]:
# Save raw embeddings
np.save(OUTPUT_EMBEDDINGS, embeddings)
ids_df.to_csv(OUTPUT_IDS, index=False)

# Save merged dataset with embedding columns
emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)

merged_emb = pd.concat(
    [ids_df.reset_index(drop=True), emb_df.reset_index(drop=True)],
    axis=1
)

final_df = df.merge(merged_emb, on="cat_no", how="inner")
final_df.to_csv(OUTPUT_MERGED, index=False)

# Save failed paths if any
if failed_paths:
    pd.DataFrame({"failed_path": failed_paths}).to_csv(OUTPUT_FAILED, index=False)

print("\nSaved files:")
print("-", OUTPUT_EMBEDDINGS)
print("-", OUTPUT_IDS)
print("-", OUTPUT_MERGED)
if failed_paths:
    print("-", OUTPUT_FAILED)


Saved files:
- ../Embeddings/siglip/siglip_strict_embeddings.npy
- ../Embeddings/siglip/siglip_strict_catnos.csv
- ../Embeddings/siglip/final_dataset_strict_with_siglip.csv


In [21]:
print("Final merged shape:", final_df.shape)
final_df[["cat_no"] + emb_cols[:5]].head()

Final merged shape: (3060, 789)


,cat_no,emb_0,emb_1,emb_2,emb_3,emb_4
0,P000002,0.260001,0.322269,-0.537111,-0.049422,0.086726
1,P000006,-0.326987,0.363749,-0.292929,0.076120,0.699717
2,P000016,-0.769486,0.088124,0.039527,0.004418,0.239459
3,P000018,0.225344,0.600198,-0.274168,0.797436,0.292645
4,P000019,0.277954,0.395194,-0.244844,-0.254475,0.219417


In [22]:
import numpy as np
import pandas as pd

# Load outputs
emb = np.load("../Embeddings/siglip/siglip_strict_embeddings.npy")
ids = pd.read_csv("../Embeddings/siglip/siglip_strict_catnos.csv")
df = pd.read_csv("../Embeddings/siglip/final_dataset_strict_with_siglip.csv")

print("Embeddings shape:", emb.shape)
print("IDs shape:", ids.shape)
print("Merged dataset shape:", df.shape)

# Check alignment
print("\nUnique cat_no in IDs:", ids["cat_no"].nunique())
print("Unique cat_no in dataset:", df["cat_no"].nunique())

# Check embedding stats
print("\nEmbedding stats:")
print("Mean:", emb.mean())
print("Std:", emb.std())
print("Min:", emb.min())
print("Max:", emb.max())

Embeddings shape: (3060, 768)
IDs shape: (3060, 1)
Merged dataset shape: (3060, 789)

Unique cat_no in IDs: 3060
Unique cat_no in dataset: 3060

Embedding stats:
Mean: 0.036481496
Std: 0.5247488
Min: -4.100309
Max: 7.2785964
